In [ ]:
from pyspark.sql import SparkSession, column
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType, StructType, StructField, DateType, DecimalType, LongType, BooleanType, TimestampType, FloatType, NumericType



In [ ]:
spark = SparkSession.builder.appName("silver.py").master("spark://spark-master:7077") \
.config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
.config("spark.hadoop.fs.s3a.access.key", "minio") \
.config("spark.hadoop.fs.s3a.secret.key", "minio123") \
.config("spark.hadoop.fs.s3a.path.style.access", "true") \
.config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
.config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
.config("spark.hadoop.fs.s3a.fast.upload", "true") \
.config("spark.driver.extraClassPath", "/home/jovyan/extra-jars/*") \
.config("spark.executor.extraClassPath", "/opt/spark/extra-jars/*") \
.getOrCreate()

In [ ]:
spark.getActiveSession()

In [ ]:
df_original = spark.read.csv("s3a://bronze/kc_house_data.csv", header=True)

In [ ]:
df_original.printSchema()

In [ ]:
df_original.show(10)

In [ ]:
df_columns = df_original.columns

#remocao de espaços para ''
for c in df_columns:
    df_sem_espacos = df_sem_espacos.withColumn(c, F.regexp_replace(c, r'\s+',''))

#quantidade de registros com ''
df_sem_espacos.select([
    F.sum((F.col(c) == "").cast("int")).alias(c)
    for c in df_columns
]).show()

#quantidade de registro NULL ou None
df_sem_espacos.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_columns]).show()

#Validar duplicadas
df_sem_espacos.groupBy(df_columns).count().filter(F.col("count") > 1).count()


In [ ]:
#tratamento price
df_sem_espacos.select("price").describe().show()

df_preco_padroes = df_sem_espacos.select(
    "price",
    F.when(F.col("price").rlike(r"^\d+(\.\d+)?$"), "ponto")
     .when(F.col("price").rlike(r"^\d+(\,\d+)?$"), "virgula")
     .otherwise("invalido")
     .alias("padrao")
)

#separar padroes
df_preco_padroes.groupBy(F.col("padrao")).count().show()

#ver tipos de invalidos
df_preco_padroes.filter(F.col("padrao")=="invalido").select("price").show()

In [ ]:
# df_cast = df_sem_espacos \
#     .withColumn("id", F.col("id").cast(LongType()))\
#     .withColumn("date", F.to_timestamp("date", "yyyyMMdd'T'HHmmss").cast(TimestampType()))\
#     .withColumn("price", F.col("price").cast(DoubleType()))\
#     .withColumn("bathrooms", F.floor(F.col("bathrooms").cast(DoubleType())))\
#     .withColumn("bedrooms", F.col("bedrooms").cast(IntegerType()))\
#     .withColumn("sqft_living", F.col("sqft_living").cast(DoubleType()))\
#     .withColumn("sqft_lot", F.col("sqft_lot").cast(DoubleType()))\
#     .withColumn("floors", F.col("floors").cast(DecimalType()))\
#     .withColumn("waterfront", F.floor(F.col("waterfront").cast(DoubleType())).cast(IntegerType()))\
#     .withColumn("view", F.floor(F.col("view").cast(DoubleType())).cast(IntegerType()))\
#     .withColumn("condition", F.col("condition").cast(DoubleType()))\
#     .withColumn("grade", F.col("grade").cast(DoubleType()))\
#     .withColumn("sqft_above", F.col("sqft_above").cast(DoubleType()))\
#     .withColumn("sqft_basement", F.col("sqft_basement").cast(DoubleType()))\
#     .withColumn("yr_built", F.col("yr_built").cast(IntegerType()))\
#     .withColumn("yr_renovated", F.col("yr_renovated").cast(IntegerType()))\
#     .withColumn("zipcode", F.col("zipcode").cast(LongType()))\
#     .withColumn("lat", F.col("lat").cast(DoubleType()))\
#     .withColumn("long", F.col("long").cast(DoubleType()))\
#     .withColumn("sqft_living15", F.col("sqft_living15").cast(DoubleType()))\
#     .withColumn("sqft_lot15", F.col("sqft_lot15").cast(DoubleType()))\

# df_cast.show(5)
# df_cast.printSchema()